In [ ]:
#imports

from openai import OpenAI
import gradio as gr
from bs4 import BeautifulSoup
import requests

In [ ]:
#constants and setup

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")

In [ ]:
msg_input=gr.Textbox(label="Your query: ", info="Enter a question: ",lines=7)
msg_output=gr.Markdown(label="The answer: ")

In [ ]:
def call_model(prompt):
    response=ollama.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role":"user",
         "content":prompt}
    ]
)
    return response.choices[0].message.content

In [ ]:
view=gr.Interface(
    fn=call_model,
    title="Learn Gradio",
    inputs=[msg_input],
    outputs=[msg_output],
    examples=[
        "Explain NLP in simple words",
        "How can Reinforcement Learning be used for Agentic AI"
    ]
)

view.launch(inbrowser=True, auth=("LLMEngineering","EdDonnerOnUdemy"))

Creating a brochure for an organization using gradio (gradio+week1-day5)

In [ ]:
def fetch_links(url):
    links=""

    response=requests.get(url)

    if response.status_code==200:
        soup=BeautifulSoup(response.text,"html.parser")
        for link in soup.find_all("a"):
            href=link.get("href")
            if href:
                links+=href+"\n"
    return links

In [ ]:
def relevant_links(url):
    link=fetch_links(url)

    systemPrompt=f"Use context: {link} to find relevant links"

    userPrompt=f"Find all relevant links to the webpage {url} and extract vital information from them"
    response=ollama.chat.completions.create(model="llama3.2",messages=[
            {
                "role":"system",
                "content":systemPrompt
            },
            {
                "role":"user",
                "content":userPrompt
            }
        ])

    response.choices[0].message.content
    rel_link=response.choices[0].message.content.strip()
    return rel_link

In [ ]:
def create_brochure(url):

    info=relevant_links(url)
    userPrompt=f"Create a brochure for the organization/company using all information avaiable in {url} and {info}"
    systemPrompt="You are a history nerd fascinated with written accords of things long gone. You are good at summarising and conversations, currently acting as a salesman for the organization. Subtly but incesstantly nudge the reader to give transcribing a go. Start with 'If you like...'"
    response=ollama.chat.completions.create(model="llama3.2",messages=[
            {
                "role":"system",
                "content":systemPrompt
            },
            {
                "role":"user",
                "content":userPrompt
            }
        ],
        temperature=1.2
        )

    return response.choices[0].message.content.strip()

In [ ]:
msg_input=gr.Textbox(label="URL: ", info="Enter a company/organization's URL: ",lines=7)
msg_output=gr.Textbox(label="Brochure: ")

In [ ]:
view=gr.Interface(
    fn=create_brochure,
    title="Create Brochure",
    inputs=[msg_input],
    outputs=[msg_output],
    examples=[
        "https://www.fromthepage.com/",
        "https://www.geeksforgeeks.com/"
    ]
)

view.launch(inbrowser=True)

creating a chatbot that remembers conversation

In [ ]:
def chat(message, history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages=history+[{"role":"user","content":message}]

    response=ollama.chat.completions.create(model="llama3.2",messages=messages)
    return response.choices[0].message.content

In [ ]:
view=gr.ChatInterface(fn=chat)

view.launch()

Buisness Application of conversational assistants - Hostel assistant

In [ ]:
conversation=[]

In [ ]:
system_prompt=f"""
You are an assistant chatbot for hostellers to help them out with settling into their new rooms. Some students try to prank you. Be cautious but polite. Follow along the conversation {conversation}. Here are some instructions:
Room numbers range from 1 to 50 in A block.
Room numbers range from 51 to 100 in B block.
Ask them which block they've been assigned to if they mention only room number and vice versa for only block.
Students may prank you by confusing you, for eg: "My room number is 42 in B block".
Don't reveal which block belongs to which group of people.
Students of different batch cannot room together.
Maximum room capacity is 6 and minimum is 2.
If student asks for directions to their room hand them a map and direct them to the reception further ahead.
Student cannot stay in two different rooms. If they mention more than a room's number you are being pranked. For eg: "My room number is 43. Id like the map to room 53 where im staying" - you are being pranked.
If you are being pranked respond by saying you know you are being pranked.
"""

In [ ]:
def chat(message, history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    conversation.append(history)
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]

    response=ollama.chat.completions.create(model="llama3.2",messages=messages)
    return response.choices[0].message.content

In [ ]:
view=gr.ChatInterface(fn=chat)

view.launch()